<a href="https://colab.research.google.com/github/jobayer360s/Data-mining-and-Data-warehouse-Project/blob/main/1.ai_and_ds_job_salary_pred_by_data_mininnig.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Importing Necessary Libaries

In [ ]:
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
import torch.nn as nn
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# 2 Load Data set

Loding the dataset before EDA and for data understanding

In [ ]:
import kagglehub
path = kagglehub.dataset_download("uditjain13/ai-and-data-science-job-salaries-2026")
print("Path to dataset files:", path)


In [ ]:
# df = pd.read_csv("/kaggle/input/datasets/uditjain13/ai-and-data-science-job-salaries-2026/ai_ds_job_salaries_2026.csv")
df = pd.read_csv("/kaggle/input/ai-and-data-science-job-salaries-2026/ai_ds_job_salaries_2026.csv") #colab
df.head(5)

In [ ]:
df.tail(5)

# 3 Perform EDA .
To Understand data  we have performing EDA exploratory data analysis EDA

In [ ]:
print('DataFrame Info:')
df.info()

print('\nMissing values per column:')
print(df.isnull().sum())

print('\nDescriptive statistics:')
df.describe().T

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
print("Dataset shape ",df.shape)
print("Duplicate ",df.duplicated().sum())

In [ ]:
print("Job Titles:", df['job_title'].nunique())
print(df['job_title'].value_counts())

## Salary Distribution


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['salary_usd'], bins=40, kde=True, color='teal')
plt.title("Distribution of Salary (USD)",fontweight='bold')
plt.xlabel("Salary (USD)")
plt.show()

print(df['salary_usd'].describe())

In [ ]:
plt.figure(figsize=(10,5))
order = df.groupby('experience_level')['salary_usd'].median().sort_values().index
sns.boxplot(data=df, x='experience_level', y='salary_usd', order=order)
plt.title("Salary by Experience Level",fontweight='bold')
plt.xlabel("Experience Level")
plt.ylabel("Salary (USD)")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
order = df.groupby('job_title')['salary_usd'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='job_title', y='salary_usd', order=order)
plt.title("Salary by Job Title",fontweight='bold')
plt.xticks(rotation=60, ha='right')
plt.xlabel("")
plt.ylabel("Salary (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# Salary Boxplot
fig = px.box(
    df,
    y='salary_usd',
    title='Salary Distribution and Potential Outliers',
    labels={'salary_usd': 'Salary (USD)'}
)

fig.show()

 IQR Calculation
* Calculate IQR for salary
*  Identify potential outliers

In [ ]:
Q1 = df['salary_usd'].quantile(0.25)
Q3 = df['salary_usd'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1          : ${Q1:,.2f}")
print(f"Q3          : ${Q3:,.2f}")
print(f"IQR         : ${IQR:,.2f}")
print(f"Lower Bound : ${lower_bound:,.2f}")
print(f"Upper Bound : ${upper_bound:,.2f}")

In [ ]:
outliers = df[
    (df['salary_usd'] < lower_bound) |
    (df['salary_usd'] > upper_bound)
]

print(f"Potential outliers: {len(outliers)}")
print(f"Percentage: {len(outliers) / len(df) * 100:.2f}%")

In [ ]:
outliers.sort_values('salary_usd', ascending=False).head(20)

### Outlier Analysis

The IQR-based analysis identified 103 observations (2.06% of the dataset)
as potential high-salary outliers. The upper bound was approximately
usd 240,USD 755, while the maximum observed salary was approximately usd 372, usd 347.
Since these observations are associated with various legitimate AI and
Data Science job roles, they were not removed automatically. Their impact
will be considered during the preprocessing and modeling stages.

# 4 Feature Audit and Selection

Before building the final preprocessing pipeline, the predictor variables are
reviewed to determine their data types, uniqueness, and missing values. This
step helps identify suitable features for salary prediction and supports the
design of the subsequent data mining and data warehouse components.

In [ ]:
# Feature audit summary

feature_audit = pd.DataFrame({
    'Data Type': df.drop(columns=['salary_usd']).dtypes.astype(str),
    'Unique Values': df.drop(columns=['salary_usd']).nunique(),
    'Missing Values': df.drop(columns=['salary_usd']).isnull().sum()
})

feature_audit

In [ ]:
df['salary_currency'].value_counts()

### Checking the relationship between company location and salary currency

In [ ]:


currency_location = pd.crosstab(
    df['company_location'],
    df['salary_currency']
)

currency_location

### Feature Audit: Salary Currency

The `salary_currency` feature was examined to determine whether it provides
independent information for salary prediction. A cross-tabulation between
`company_location` and `salary_currency` shows a strong one-to-one or
one-to-many geographical correspondence, with currencies largely determined
by the company location.

Since the target variable `salary_usd` is already standardized to US dollars,
the `salary_currency` feature provides limited additional predictive
information and may introduce redundant information alongside
`company_location`. Therefore, `salary_currency` is excluded from the final
predictor set.

In [ ]:
# Drop redundant feature

df_model = df.drop(columns=['salary_currency'])

print("Original shape:", df.shape)
print("Shape after removing salary_currency:", df_model.shape)

In [ ]:
# Compare company location and employee residence using percentages

location_pct = pd.crosstab(
    df['company_location'],
    df['employee_residence'],
    normalize='index'
) * 100

plt.figure(figsize=(10, 6))

sns.heatmap(
    location_pct,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar_kws={'label': 'Percentage (%)'}
)

plt.title(
    'Company Location vs. Employee Residence',
    fontweight='bold'
)

plt.xlabel('Employee Residence')
plt.ylabel('Company Location')
plt.tight_layout()
plt.show()

### Location Feature Analysis

The heatmap shows a strong geographical alignment between company location
and employee residence, with most observations concentrated along the
diagonal. However, the presence of non-diagonal observations indicates that
employees may reside in countries different from their company locations.
Therefore, the two features are considered related but not fully redundant
and are retained for further analysis.

In [ ]:
df['remote_ratio'].value_counts().sort_index()

### Feature Audit: Remote Work Ratio

The `remote_ratio` feature contains three distinct values: 0, 50, and 100,
representing different levels of remote work. The distribution shows that
all three categories are sufficiently represented in the dataset. Since the
values have a meaningful ordered interpretation as the proportion of remote
work, the feature is retained as a numerical predictor for further analysis.

In [ ]:
df['years_experience'].describe()

In [ ]:
# Compare experience level with years of experience

experience_summary = (
    df.groupby('experience_level')['years_experience']
      .agg(['count', 'mean', 'median', 'min', 'max'])
      .sort_values('median')
)

experience_summary

# 5 Corelation Heatmap -

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10,8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Heatmap (Numeric Features)",fontweight='bold')
plt.show()

## 5.1 AI Adoption & Salary Deep-Dive

This project's dataset includes several features not typically found in salary-prediction studies: `ai_tools_hours_per_week`, `uses_ai_tools_daily`, `fears_ai_automation_score`, and `upskilling_hours_per_month`. This section examines whether AI-tool adoption and automation anxiety are associated with compensation, independent of experience level.

### 5.1.1 Correlation and Group Comparison

In [ ]:
from scipy import stats

# --- Basic correlation: AI tool usage hours vs salary ---
corr_ai_hours = df['ai_tools_hours_per_week'].corr(df['salary_usd'])
print(f"Correlation (ai_tools_hours_per_week vs salary_usd): {corr_ai_hours:.3f}")

corr_fear = df['fears_ai_automation_score'].corr(df['salary_usd'])
print(f"Correlation (fears_ai_automation_score vs salary_usd): {corr_fear:.3f}")

corr_upskill = df['upskilling_hours_per_month'].corr(df['salary_usd'])
print(f"Correlation (upskilling_hours_per_month vs salary_usd): {corr_upskill:.3f}")

# --- T-test: does daily AI tool usage split the salary distribution? ---
uses_ai = df[df['uses_ai_tools_daily'] == True]['salary_usd']
no_ai = df[df['uses_ai_tools_daily'] == False]['salary_usd']

t_stat, p_val = stats.ttest_ind(uses_ai, no_ai, equal_var=False)
print(f"\nMean salary (uses AI daily): {uses_ai.mean():,.0f}")
print(f"Mean salary (doesn't use AI daily): {no_ai.mean():,.0f}")
print(f"T-test: t={t_stat:.3f}, p={p_val:.4f}")

### 5.1.2 Controlling for Experience — Is the AI-Usage Salary Gap Real?

In [ ]:
import statsmodels.formula.api as smf

# OLS regression controlling for years_experience and experience_level
ai_adoption_model = smf.ols(
    'salary_usd ~ uses_ai_tools_daily + ai_tools_hours_per_week + '
    'fears_ai_automation_score + years_experience + C(experience_level)',
    data=df
).fit()

print(ai_adoption_model.summary())
# The coefficient and p-value on uses_ai_tools_daily and ai_tools_hours_per_week
# show whether the effect survives after controlling for experience.

### 5.1.3 Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Salary by daily AI usage
df.boxplot(column='salary_usd', by='uses_ai_tools_daily', ax=axes[0])
axes[0].set_title('Salary by Daily AI Tool Usage')
axes[0].set_xlabel('Uses AI Tools Daily')
axes[0].set_ylabel('Salary (USD)')

# 2. Scatter: AI tool hours vs salary, colored by experience level
for level in df['experience_level'].unique():
    subset = df[df['experience_level'] == level]
    axes[1].scatter(subset['ai_tools_hours_per_week'], subset['salary_usd'],
                     alpha=0.4, label=level, s=15)
axes[1].set_xlabel('AI Tool Hours per Week')
axes[1].set_ylabel('Salary (USD)')
axes[1].set_title('AI Tool Usage vs Salary by Experience')
axes[1].legend(fontsize=8)

# 3. Fear of automation vs job satisfaction
axes[2].scatter(df['fears_ai_automation_score'], df['job_satisfaction_score'], alpha=0.3)
axes[2].set_xlabel('Fears AI Automation Score')
axes[2].set_ylabel('Job Satisfaction Score')
axes[2].set_title('AI Anxiety vs Job Satisfaction')

plt.suptitle('')
plt.tight_layout()
plt.show()

# 6 Data Preprocessing


## 6.1 Define target and feature

In [ ]:

target = 'salary_usd'

# Remove target and redundant feature
X = df.drop(columns=[target, 'salary_currency'])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 6.2 Identify categorical and numerical columns

In [ ]:
cat_cols = X.select_dtypes(
    include=['object', 'bool']
).columns.tolist()

num_cols = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print("Categorical features:")
print(cat_cols)

print("\nNumerical features:")
print(num_cols)

## 6.3 Train Test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 6.4 Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            cat_cols
        )
    ],
    remainder='passthrough'
)
print("Done")

## 6.7 Final Pre processing Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


target = 'salary_usd'

X = df.drop(columns=[target, 'salary_currency'])
y = df[target]




cat_cols = X.select_dtypes(
    include=['object', 'bool']
).columns.tolist()

num_cols = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()


print("Number of categorical features:", len(cat_cols))
print("Number of numerical features:", len(num_cols))



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            cat_cols
        )
    ],
    remainder='passthrough'
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

# 7 Model Building

## 7.1 Define Candidate Models

* Linear Regression
* baseline modelRandom Forest
*  ensemble tree modelXGBoost
*  gradient boostingLightGBM
*  gradient boosting
* cat Boost
* Extra Trees


In [ ]:
!pip install catboost

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {

    'Linear Regression': LinearRegression(),

    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    'XGBoost': XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        n_jobs=-1
    ),

    'LightGBM': LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    'CatBoost': CatBoostRegressor(
        iterations=400,
        learning_rate=0.05,
        depth=5,
        random_seed=42,
        verbose=False
    )
}

## 7.2 Train and Generate Prediction

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

results = []
fitted_pipelines = {}

for name, model in models.items():

    # Create preprocessing + model pipeline
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Train model
    pipe.fit(X_train, y_train)

    # Predict on unseen test data
    preds = pipe.predict(X_test)

    # Evaluation metrics
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5

    # Store results
    results.append({
        'Model': name,
        'R2 Score': r2,
        'MAE': mae,
        'RMSE': rmse
    })

    # Store fitted pipeline for later analysis
    fitted_pipelines[name] = pipe

## 7.3 Tree Vizualization

In [ ]:
# Get feature names after preprocessing
feature_names = (
    fitted_pipelines['Random Forest']
    .named_steps['preprocessor']
    .get_feature_names_out()
)

print("Total transformed features:", len(feature_names))

### 7.3.1 Random Forest Vizualization

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Extract trained Random Forest model
rf_model = fitted_pipelines['Random Forest'].named_steps['model']

# Select the first tree
rf_tree = rf_model.estimators_[0]

plt.figure(figsize=(24, 12))

plot_tree(
    rf_tree,
    feature_names=feature_names,
    max_depth=3,
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title(
    "Representative Decision Tree from Random Forest",
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

### 7.3.2 XGBoost — Representative Tree

In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt

# Extract trained XGBoost model
xgb_model = fitted_pipelines['XGBoost'].named_steps['model']

# Create figure and axes explicitly
fig, ax = plt.subplots(figsize=(24, 12))

# Plot the first tree
xgb.plot_tree(
    xgb_model,
    tree_idx=0,
    rankdir='LR',
    ax=ax,
    with_stats=True
)

ax.set_title(
    "Representative Decision Tree from XGBoost",
    fontsize=16,
    fontweight='bold',
    pad=20
)

plt.tight_layout()
plt.show()

### 7.3.3 LightGBM — Representative Tree

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt

# Extract trained LightGBM model
lgb_model = fitted_pipelines['LightGBM'].named_steps['model']

# Create figure and axes explicitly
fig, ax = plt.subplots(figsize=(24, 12))

# Plot the first tree
lgb.plot_tree(
    lgb_model,
    tree_index=0,
    ax=ax,
    figsize=(24, 12),
    show_info=[
        'split_gain',
        'internal_value',
        'internal_count'
    ]
)

ax.set_title(
    "Representative Decision Tree from LightGBM",
    fontsize=16,
    fontweight='bold',
    pad=20
)

plt.tight_layout()
plt.show()

## 7.4 Model Performance comparision

In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values('R2 Score', ascending=False)
    .reset_index(drop=True)
)

results_df

In [ ]:
results_display = results_df.copy()

results_display['R2 Score'] = results_display['R2 Score'].round(4)
results_display['MAE'] = results_display['MAE'].round(2)
results_display['RMSE'] = results_display['RMSE'].round(2)

results_display

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

sns.barplot(data=results_df, x='Model', y='R2 Score', ax=axes[0])
axes[0].set_title("R2 Score by Model (higher = better)",fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=results_df, x='Model', y='RMSE', ax=axes[1])
axes[1].set_title("RMSE by Model (lower = better)",fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 7.4.1 Cross-Validated Performance (Baseline Models)

The comparison above uses a single train/test split. To check that these results are stable, we re-evaluate all baseline models using 5-fold cross-validation on the full dataset.

In [ ]:
from sklearn.model_selection import cross_validate, KFold
import pandas as pd

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results_baseline = {}

for name, pipe in fitted_pipelines.items():
    scores = cross_validate(
        pipe, X, y, cv=kf,
        scoring=['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error'],
        n_jobs=-1
    )
    cv_results_baseline[name] = {
        'R2_mean': scores['test_r2'].mean(),
        'R2_std': scores['test_r2'].std(),
        'RMSE_mean': -scores['test_neg_root_mean_squared_error'].mean(),
        'RMSE_std': scores['test_neg_root_mean_squared_error'].std(),
        'MAE_mean': -scores['test_neg_mean_absolute_error'].mean(),
        'MAE_std': scores['test_neg_mean_absolute_error'].std(),
        'raw_r2_scores': scores['test_r2']
    }

cv_df_baseline = pd.DataFrame(cv_results_baseline).T
cv_df_baseline[['R2_mean', 'R2_std', 'RMSE_mean', 'RMSE_std', 'MAE_mean', 'MAE_std']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
models_list = cv_df_baseline.index
ax.bar(models_list, cv_df_baseline['R2_mean'], yerr=cv_df_baseline['R2_std'], capsize=5)
ax.set_ylabel('R2 Score')
ax.set_title('5-Fold Cross-Validated R2 - Baseline Models')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 7.5 Feature Importance (Best Model)
Which features drive the model's salary predictions the most?

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipe = fitted_pipelines[best_model_name]

feature_names = best_pipe.named_steps['preprocessor'].get_feature_names_out()
importances = best_pipe.named_steps['model'].feature_importances_

fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(9,6))
sns.barplot(data=fi_df, x='importance', y='feature')
plt.title(f"Top 15 Feature Importances ({best_model_name})",fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt


best_pipeline = fitted_pipelines['LightGBM']

lgb_model = best_pipeline.named_steps['model']

preprocessor_fitted = best_pipeline.named_steps['preprocessor']

transformed_feature_names = (
    preprocessor_fitted.get_feature_names_out()
)


importance_values = lgb_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': transformed_feature_names,
    'Importance': importance_values
})

feature_importance_df = (
    feature_importance_df
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

feature_importance_df.head(15)

## 7.5.1 Aggregate Importance by Original Feature

In [ ]:
# Get fitted preprocessing components
preprocessor_fitted = (
    fitted_pipelines['LightGBM']
    .named_steps['preprocessor']
)

encoder = preprocessor_fitted.named_transformers_['categorical']

# Get all transformed feature names
encoded_feature_names = encoder.get_feature_names_out(cat_cols)

# Get LightGBM feature importances
lgb_model = fitted_pipelines['LightGBM'].named_steps['model']

importance_values = lgb_model.feature_importances_

# Get all transformed feature names from the full preprocessor
transformed_feature_names = (
    preprocessor_fitted.get_feature_names_out()
)

# Create mapping
feature_mapping = []

# Map one-hot encoded categorical features
for encoded_name in encoded_feature_names:

    original_feature = None

    for cat in cat_cols:
        if encoded_name.startswith(cat + "_"):
            original_feature = cat
            break

    feature_mapping.append({
        'Transformed Feature': 'categorical__' + encoded_name,
        'Original Feature': original_feature
    })

# Map numerical features
for feature in num_cols:
    feature_mapping.append({
        'Transformed Feature': 'remainder__' + feature,
        'Original Feature': feature
    })

mapping_df = pd.DataFrame(feature_mapping)

# Create importance dataframe
importance_df = pd.DataFrame({
    'Transformed Feature': transformed_feature_names,
    'Importance': importance_values
})

# Merge
importance_mapped = importance_df.merge(
    mapping_df,
    on='Transformed Feature',
    how='left'
)

# Aggregate by original feature
aggregated_importance = (
    importance_mapped
    .groupby('Original Feature', as_index=False)['Importance']
    .sum()
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

aggregated_importance.head(15)

### 7.5.2 Plot Top 15 Original Features

In [ ]:
# Top 15 original features
top_features = aggregated_importance.head(15).sort_values(
    'Importance',
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features['Original Feature'],
    top_features['Importance']
)

plt.xlabel('Aggregated Feature Importance')
plt.ylabel('Feature')
plt.title(
    'Top 15 Features Influencing AI and Data Science Job Salary Prediction',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
top_importance_table = aggregated_importance.head(15).copy()

top_importance_table['Importance (%)'] = (
    top_importance_table['Importance']
    / aggregated_importance['Importance'].sum()
    * 100
).round(2)

top_importance_table = top_importance_table[
    ['Original Feature', 'Importance', 'Importance (%)']
]

top_importance_table

# 8. Hyperparameter Tuning


Hyperparameter tuning was performed to optimize the performance of the two best baseline ensemble models, XGBoost and LightGBM. RandomizedSearchCV with 3-fold cross-validation was used to search for promising hyperparameter combinations using only the training data. The test set was kept completely unseen during the tuning process.


The hyperparameter tuning process follows the workflow below:

```text
Training Data
      │
      ├── 3-Fold Cross-Validation
      │
      ├── Hyperparameter Search
      │
      └── Best Parameters
                │
                ▼
          Tuned Model
                │
                ▼
          Final Test Set

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

### 8.3 Tune Light BGM

In [ ]:
# Extract the preprocessing pipeline
preprocessor_for_tuning = preprocessor

# LightGBM pipeline
lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_for_tuning),
    ('model', LGBMRegressor(
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ))
])

In [ ]:
lgbm_param_dist = {
    'model__n_estimators': randint(200, 800),
    'model__learning_rate': uniform(0.01, 0.09),
    'model__max_depth': randint(3, 10),
    'model__num_leaves': randint(15, 80),
    'model__min_child_samples': randint(10, 50),
    'model__subsample': uniform(0.7, 0.3),
    'model__colsample_bytree': uniform(0.7, 0.3)
}

### 8.4 Run LightGBM Randomized Search

In [ ]:
lgbm_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=lgbm_param_dist,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)

### 8.5 Best LightGBM Parameters

In [ ]:
print("Best LightGBM Parameters:")
print(lgbm_search.best_params_)

print("\nBest Cross-Validation RMSE:")
print(-lgbm_search.best_score_)

### 8.6 Tune XG Boost

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective='reg:squarederror'
    ))
])

In [ ]:
xgb_param_dist = {
    'model__n_estimators': randint(200, 800),
    'model__learning_rate': uniform(0.01, 0.09),
    'model__max_depth': randint(3, 10),
    'model__min_child_weight': randint(1, 10),
    'model__subsample': uniform(0.7, 0.3),
    'model__colsample_bytree': uniform(0.7, 0.3),
    'model__gamma': uniform(0, 0.5)
}

In [ ]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

In [ ]:
print("Best XGBoost Parameters:")
print(xgb_search.best_params_)

print("\nBest Cross-Validation RMSE:")
print(-xgb_search.best_score_)

### 8.10 Tune Extra Trees

In [ ]:
et_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
et_param_dist = {
    'model__n_estimators': randint(200, 800),
    'model__max_features': ['sqrt', 'log2', 0.5, 0.7, 0.9],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf': randint(1, 20),
    'model__bootstrap': [True, False]
}

### 8.11 Run Extra Trees Randomized Search

In [ ]:
et_search = RandomizedSearchCV(
    estimator=et_pipeline,
    param_distributions=et_param_dist,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

et_search.fit(X_train, y_train)

### 8.12 Best Extra Trees Parameters

In [ ]:
print("Best Extra Trees Parameters:")
print(et_search.best_params_)

print("\nBest Cross-Validation RMSE:")
print(-et_search.best_score_)

### 8.7 Tune CatBoost

In [ ]:
catboost_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', CatBoostRegressor(
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    ))
])

In [ ]:
catboost_param_dist = {
    'model__iterations': randint(200, 800),
    'model__learning_rate': uniform(0.01, 0.09),
    'model__depth': randint(3, 10),
    'model__l2_leaf_reg': uniform(1, 10),
    'model__subsample': uniform(0.7, 0.3)
}

### 8.8 Run CatBoost Randomized Search

In [ ]:
catboost_search = RandomizedSearchCV(
    estimator=catboost_pipeline,
    param_distributions=catboost_param_dist,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

catboost_search.fit(X_train, y_train)

### 8.9 Best CatBoost Parameters

In [ ]:
print("Best CatBoost Parameters:")
print(catboost_search.best_params_)

print("\nBest Cross-Validation RMSE:")
print(-catboost_search.best_score_)

### 9.1 Evaluate Tuned Models

In [ ]:
tuned_models = {
    'Tuned LightGBM': lgbm_search.best_estimator_,
    'Tuned XGBoost': xgb_search.best_estimator_,
    'Tuned CatBoost': catboost_search.best_estimator_,
    'Tuned Extra Trees': et_search.best_estimator_
}

tuned_results = [] # Re-initialize tuned_results to avoid appending to previous runs
tuned_pipelines = {} # Re-initialize tuned_pipelines


for name, pipe in tuned_models.items():

    # Predict on untouched test set
    preds = pipe.predict(X_test)

    # Calculate evaluation metrics
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5

    tuned_results.append({
        'Model': name,
        'R2 Score': r2,
        'MAE': mae,
        'RMSE': rmse
    })

    tuned_pipelines[name] = pipe

tuned_results_df = (
    pd.DataFrame(tuned_results)
    .sort_values('R2 Score', ascending=False)
    .reset_index(drop=True)
)

display(tuned_results_df)

### Comparison: Tuned Extra Trees vs. Tuned CatBoost

In [ ]:
extra_trees_vs_catboost_comparison = tuned_results_df[
    tuned_results_df['Model'].isin(['Tuned Extra Trees', 'Tuned CatBoost'])
].sort_values(by='R2 Score', ascending=False).reset_index(drop=True)
display(extra_trees_vs_catboost_comparison)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=extra_trees_vs_catboost_comparison, x='Model', y='R2 Score', palette='viridis')
plt.title('R2 Score Comparison: Tuned Extra Trees vs. Tuned CatBoost', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('R2 Score')
plt.ylim(0, 1) # R2 score is between 0 and 1
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=extra_trees_vs_catboost_comparison, x='Model', y='MAE', palette='magma')
plt.title('MAE Comparison: Tuned Extra Trees vs. Tuned CatBoost (Lower is Better)', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('MAE')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=extra_trees_vs_catboost_comparison, x='Model', y='RMSE', palette='rocket')
plt.title('RMSE Comparison: Tuned Extra Trees vs. Tuned CatBoost (Lower is Better)', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('RMSE')
plt.tight_layout()
plt.show()

In [ ]:
tuned_models = {
    'Tuned LightGBM': lgbm_search.best_estimator_,
    'Tuned XGBoost': xgb_search.best_estimator_,
    'Tuned CatBoost': catboost_search.best_estimator_,
    'Tuned Extra Trees': et_search.best_estimator_
}

tuned_results = [] # Re-initialize tuned_results to avoid appending to previous runs
tuned_pipelines = {} # Re-initialize tuned_pipelines

for name, pipe in tuned_models.items():

    # Predict on untouched test set
    preds = pipe.predict(X_test)

    # Calculate evaluation metrics
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5

    tuned_results.append({
        'Model': name,
        'R2 Score': r2,
        'MAE': mae,
        'RMSE': rmse
    })

    tuned_pipelines[name] = pipe

tuned_results_df = (
    pd.DataFrame(tuned_results)
    .sort_values('R2 Score', ascending=False)
    .reset_index(drop=True)
)

display(tuned_results_df)

## 9.2 Baseline vs Tuned Model Comparison

In [ ]:
baseline_comparison = results_df[
    ['Model', 'R2 Score', 'MAE', 'RMSE']
].copy()

tuned_comparison = tuned_results_df.loc[
    tuned_results_df['Model'].isin(['Tuned LightGBM', 'Tuned XGBoost', 'Tuned CatBoost'])
][['Model', 'R2 Score', 'MAE', 'RMSE']].copy()

comparison_df = pd.concat(
    [baseline_comparison, tuned_comparison],
    ignore_index=True
)

comparison_df

In [ ]:
baseline_comparison = results_df[
    ['Model', 'R2 Score', 'MAE', 'RMSE']
].copy()

tuned_comparison = tuned_results_df.loc[
    tuned_results_df['Model'].isin([
        'Tuned LightGBM',
        'Tuned XGBoost',
        'Tuned CatBoost',
        'Tuned Extra Trees' # Include Extra Trees
    ])
][['Model', 'R2 Score', 'MAE', 'RMSE']].copy()

comparison_df = pd.concat(
    [baseline_comparison, tuned_comparison],
    ignore_index=True
)

comparison_df

##  9.3 Improvement Calculation

In [ ]:
baseline_lgbm = results_df.loc[
    results_df['Model'] == 'LightGBM'
].iloc[0]

baseline_xgb = results_df.loc[
    results_df['Model'] == 'XGBoost'
].iloc[0]

baseline_catboost = results_df.loc[
    results_df['Model'] == 'CatBoost'
].iloc[0]

baseline_et = results_df.loc[
    results_df['Model'] == 'Extra Trees'
].iloc[0] # Get baseline for Extra Trees

tuned_lgbm = tuned_results_df.loc[
    tuned_results_df['Model'] == 'Tuned LightGBM'
].iloc[0]

tuned_xgb = tuned_results_df.loc[
    tuned_results_df['Model'] == 'Tuned XGBoost'
].iloc[0]

tuned_catboost = tuned_results_df.loc[
    tuned_results_df['Model'] == 'Tuned CatBoost'
].iloc[0]

tuned_et = tuned_results_df.loc[
    tuned_results_df['Model'] == 'Tuned Extra Trees'
].iloc[0] # Get tuned for Extra Trees


improvement_df = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost', 'Extra Trees'], # Add Extra Trees

    'R2 Improvement': [
        tuned_lgbm['R2 Score'] - baseline_lgbm['R2 Score'],
        tuned_xgb['R2 Score'] - baseline_xgb['R2 Score'],
        tuned_catboost['R2 Score'] - baseline_catboost['R2 Score'],
        tuned_et['R2 Score'] - baseline_et['R2 Score'] # Add Extra Trees
    ],

    'MAE Improvement': [
        baseline_lgbm['MAE'] - tuned_lgbm['MAE'],
        baseline_xgb['MAE'] - tuned_xgb['MAE'],
        baseline_catboost['MAE'] - tuned_catboost['MAE'],
        baseline_et['MAE'] - tuned_et['MAE'] # Add Extra Trees
    ],

    'RMSE Improvement': [
        baseline_lgbm['RMSE'] - tuned_lgbm['RMSE'],
        baseline_xgb['RMSE'] - tuned_xgb['RMSE'],
        baseline_catboost['RMSE'] - tuned_catboost['RMSE'],
        baseline_et['RMSE'] - tuned_et['RMSE'] # Add Extra Trees
    ]
})

display(improvement_df)

## 9.3.1 Cross-Validated Tuned Model Comparison

The claim that Tuned CatBoost is the best model rests on a single train/test split so far. Here we re-evaluate the tuned models with 5-fold cross-validation and test whether CatBoost's edge over LightGBM and XGBoost is statistically significant.

In [ ]:
cv_results_tuned = {}

for name, pipe in tuned_pipelines.items():
    scores = cross_validate(
        pipe, X, y, cv=kf,
        scoring=['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error'],
        n_jobs=-1
    )
    cv_results_tuned[name] = {
        'R2_mean': scores['test_r2'].mean(),
        'R2_std': scores['test_r2'].std(),
        'RMSE_mean': -scores['test_neg_root_mean_squared_error'].mean(),
        'RMSE_std': scores['test_neg_root_mean_squared_error'].std(),
        'MAE_mean': -scores['test_neg_mean_absolute_error'].mean(),
        'MAE_std': scores['test_neg_mean_absolute_error'].std(),
        'raw_r2_scores': scores['test_r2']
    }

cv_df_tuned = pd.DataFrame(cv_results_tuned).T
cv_df_tuned[['R2_mean', 'R2_std', 'RMSE_mean', 'RMSE_std', 'MAE_mean', 'MAE_std']]

### Statistical Significance — Is Tuned CatBoost Actually Better?

In [ ]:
from scipy.stats import ttest_rel

catboost_scores = cv_results_tuned['Tuned CatBoost']['raw_r2_scores']
lightgbm_scores = cv_results_tuned['Tuned LightGBM']['raw_r2_scores']
xgboost_scores = cv_results_tuned['Tuned XGBoost']['raw_r2_scores']

t1, p1 = ttest_rel(catboost_scores, lightgbm_scores)
t2, p2 = ttest_rel(catboost_scores, xgboost_scores)

print(f"Tuned CatBoost vs Tuned LightGBM: t={t1:.3f}, p={p1:.4f}")
print(f"Tuned CatBoost vs Tuned XGBoost:  t={t2:.3f}, p={p2:.4f}")
print("(p < 0.05 means the difference is statistically significant, not just noise)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
models_list = cv_df_tuned.index
ax.bar(models_list, cv_df_tuned['R2_mean'], yerr=cv_df_tuned['R2_std'], capsize=5)
ax.set_ylabel('R2 Score')
ax.set_title('5-Fold Cross-Validated R2 - Tuned Models')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 9.4 Visualize Final Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# R2 Score Plot
sns.barplot(data=comparison_df, x='Model', y='R2 Score', ax=axes[0], palette='viridis')
axes[0].set_title('R2 Score (Higher is Better)', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

# MAE Plot
sns.barplot(data=comparison_df, x='Model', y='MAE', ax=axes[1], palette='magma')
axes[1].set_title('MAE (Lower is Better)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

# RMSE Plot
sns.barplot(data=comparison_df, x='Model', y='RMSE', ax=axes[2], palette='cividis')
axes[2].set_title('RMSE (Lower is Better)', fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 9.5 R2 Score Comparison of All Models

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_df, x='Model', y='R2 Score', palette='viridis')
plt.title('R2 Score Comparison Across All Models (Higher is Better)', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('R2 Score')
plt.ylim(0, 1) # R2 score is between 0 and 1
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9.6 Feature Importance for Tuned CatBoost Model

In [ ]:
# Get the best tuned CatBoost pipeline
best_tuned_catboost_pipeline = catboost_search.best_estimator_

# Extract the CatBoost model and preprocessor
catboost_model = best_tuned_catboost_pipeline.named_steps['model']
preprocessor_fitted_catboost = best_tuned_catboost_pipeline.named_steps['preprocessor']

# Get all transformed feature names
transformed_feature_names_catboost = (
    preprocessor_fitted_catboost.get_feature_names_out()
)

# Get CatBoost feature importances
importance_values_catboost = catboost_model.get_feature_importance()

# Create mapping (re-using cat_cols and num_cols from earlier preprocessing)
feature_mapping_catboost = []

# Map one-hot encoded categorical features
encoder_catboost = preprocessor_fitted_catboost.named_transformers_['categorical']
encoded_feature_names_catboost = encoder_catboost.get_feature_names_out(cat_cols)

for encoded_name in encoded_feature_names_catboost:
    original_feature = None
    for cat in cat_cols:
        if encoded_name.startswith(cat + "_"):
            original_feature = cat
            break
    feature_mapping_catboost.append({
        'Transformed Feature': 'categorical__' + encoded_name,
        'Original Feature': original_feature
    })

# Map numerical features
for feature in num_cols:
    feature_mapping_catboost.append({
        'Transformed Feature': 'remainder__' + feature,
        'Original Feature': feature
    })

mapping_df_catboost = pd.DataFrame(feature_mapping_catboost)

# Create importance dataframe
importance_df_catboost = pd.DataFrame({
    'Transformed Feature': transformed_feature_names_catboost,
    'Importance': importance_values_catboost
})

# Merge
importance_mapped_catboost = importance_df_catboost.merge(
    mapping_df_catboost,
    on='Transformed Feature',
    how='left'
)

# Aggregate by original feature
aggregated_importance_catboost = (
    importance_mapped_catboost
    .groupby('Original Feature', as_index=False)['Importance']
    .sum()
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

# Top 15 original features
top_features_catboost = aggregated_importance_catboost.head(15).sort_values(
    'Importance',
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features_catboost['Original Feature'],
    top_features_catboost['Importance']
)

plt.xlabel('Aggregated Feature Importance')
plt.ylabel('Feature')
plt.title(
    'Top 15 Features Influencing AI and Data Science Job Salary Prediction (Tuned CatBoost)',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

display(aggregated_importance_catboost.head(15))

## 9.7 Residual Analysis for Tuned CatBoost Model

In [ ]:
# Select the tuned CatBoost model
tuned_catboost_pipeline = tuned_pipelines['Tuned CatBoost']

# Make predictions on the test set
catboost_predictions = tuned_catboost_pipeline.predict(X_test)

# Calculate residuals
catboost_residuals = y_test - catboost_predictions

# Create a DataFrame for residuals for easier plotting
catboost_residuals_df = pd.DataFrame({
    'Actual Salary': y_test.values,
    'Predicted Salary': catboost_predictions,
    'Residual': catboost_residuals
})

# Plot Residual Distribution
plt.figure(figsize=(9, 6))

plt.hist(
    catboost_residuals_df['Residual'],
    bins=30,
    edgecolor='black'
)

plt.axvline(
    0,
    linestyle='--',
    color='red',
    label='Zero Residual'
)

plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title(
    'Residual Distribution — Tuned CatBoost Model',
    fontweight='bold'
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    catboost_residuals_df['Predicted Salary'],
    catboost_residuals_df['Residual'],
    alpha=0.5
)

plt.axhline(
    0,
    linestyle='--',
    color='red',
    label='Zero Residual'
)

plt.xlabel('Predicted Salary (USD)')
plt.ylabel('Residual (USD)')
plt.title(
    'Residuals vs Predicted Salary — Tuned CatBoost Model',
    fontweight='bold'
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(comparison_df.sort_values(by='R2 Score', ascending=False).reset_index(drop=True))

# 10. Prediction & Residual Analysis

### 10.1 Select Best Tuned Model

In [ ]:
best_tuned_name = tuned_results_df.iloc[0]['Model']
best_tuned_pipeline = tuned_pipelines[best_tuned_name]

print("Best Tuned Model:", best_tuned_name)

In [ ]:
final_predictions = best_tuned_pipeline.predict(X_test)

prediction_df = pd.DataFrame({
    'Actual Salary': y_test.values,
    'Predicted Salary': final_predictions
})

prediction_df.head(10)

### 10.1.1 Actual vs Predicted Salary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 7))

plt.scatter(
    y_test,
    final_predictions,
    alpha=0.5
)

# Perfect prediction line
min_salary = min(y_test.min(), final_predictions.min())
max_salary = max(y_test.max(), final_predictions.max())

plt.plot(
    [min_salary, max_salary],
    [min_salary, max_salary],
    linestyle='--'
)

plt.xlabel('Actual Salary (USD)')
plt.ylabel('Predicted Salary (USD)')
plt.title(
    f'Actual vs Predicted Salary — {best_tuned_name}',
    fontweight='bold'
)

plt.tight_layout()
plt.show()

### 10.2 Calculate Residuals

In [ ]:
prediction_df['Residual'] = (
    prediction_df['Actual Salary']
    - prediction_df['Predicted Salary']
)

prediction_df.head(10)

In [ ]:
# Residual Distribution

plt.figure(figsize=(9, 6))

plt.hist(
    prediction_df['Residual'],
    bins=30
)

plt.axvline(
    0,
    linestyle='--'
)

plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title(
    f'Residual Distribution — {best_tuned_name}',
    fontweight='bold'
)

plt.tight_layout()
plt.show()

### 10.3 Residual vs Predicted Values

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    prediction_df['Predicted Salary'],
    prediction_df['Residual'],
    alpha=0.5
)

plt.axhline(
    0,
    linestyle='--'
)

plt.xlabel('Predicted Salary (USD)')
plt.ylabel('Residual (USD)')
plt.title(
    f'Residuals vs Predicted Salary — {best_tuned_name}',
    fontweight='bold'
)

plt.tight_layout()
plt.show()